# Masiez — Train Tank Detector with Transformers (Kaggle) → ONNXThe game exports training frames from the user menu with **CAPTURE FRAME + LABELS**. Every capture is a PNG plus a COCO JSON file containing the pixel-space bounding boxes of the software-detected tanks.This notebook fine-tunes **RT-DETR** with Hugging Face Transformers on the single class **tank** and exports an ONNX graph for the browser. The software boxes are labels only; the trained model learns to detect tanks from pixels alone.## How to run on Kaggle1. Play the game, open the user menu, and press **CAPTURE FRAME + LABELS** a few dozen times across maps, distances, and angles.2. Zip the downloaded drone_frame_*.png / .json files and upload the zip as a Kaggle dataset.3. Open this notebook with a GPU accelerator, attach the dataset, and run all cells.4. Download tank-detector.onnx and tank-detector.labels.txt from the output.**Option B** — run locally from packages/notebooks with the captures in a captures/ folder. The dataset cells work everywhere; training needs a GPU to finish in reasonable time.

## 0. Install dependencies

In [ ]:
%pip install -q -U transformers torchmetrics accelerate onnx onnxruntime onnxscript

## 1. ConfigurationThe default checkpoint is the ResNet-18 RT-DETR, which is fast enough for in-browser WebGPU. Switch MODEL_ID to PekingU/rtdetr_r50vd_coco_o365 for more accuracy at a higher cost.

In [ ]:
import jsonimport osimport randomimport zipfilefrom pathlib import Pathimport torchMODEL_ID = 'PekingU/rtdetr_r18vd_coco_o365'IMG_SIZE = 640CLASSES = ['tank']ID2LABEL = {index: name for index, name in enumerate(CLASSES)}LABEL2ID = {name: index for index, name in enumerate(CLASSES)}VAL_FRACTION = 0.2SEED = 0EPOCHS = 40BATCH = 4LEARNING_RATE = 1e-4WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./tank_work')CAPTURE_ROOT = WORK / 'captures'OUTPUT_DIR = WORK / 'tank_detector'for folder in (WORK, CAPTURE_ROOT, OUTPUT_DIR):    folder.mkdir(parents=True, exist_ok=True)random.seed(SEED)torch.manual_seed(SEED)print('work dir:', WORK.resolve())

## 2. Collect capturesZips attached to the notebook are extracted before scanning. A file is treated as a capture when it contains COCO images and annotations arrays.

In [ ]:
def is_capture_json(path):    try:        data = json.loads(path.read_text())    except (OSError, json.JSONDecodeError):        return False    return isinstance(data, dict) and 'images' in data and 'annotations' in datadef unpack_archives(root):    for archive in sorted(Path(root).rglob('*.zip')):        target = CAPTURE_ROOT / archive.stem        if target.exists():            continue        target.mkdir(parents=True, exist_ok=True)        with zipfile.ZipFile(archive) as zf:            zf.extractall(target)        print('unpacked', archive, '->', target)def find_capture_json(root):    found = []    for path in sorted(Path(root).rglob('*.json')):        if '.venv' in path.parts or 'site-packages' in path.parts:            continue        if is_capture_json(path):            found.append(path)    return foundsources = []for root in (Path('/kaggle/input'), Path('captures'), CAPTURE_ROOT):    if not root.is_dir():        continue    unpack_archives(root)    sources.extend(find_capture_json(root))sources = sorted(set(sources))if not sources:    raise SystemExit('No captures found. Attach a zip of drone_frame_*.png/.json files, or put them in packages/notebooks/captures/.')print(f'{len(sources)} capture file(s)')for path in sources[:5]:    print(' ', path)

## 3. Load frames and split train/valBoxes stay in COCO pixel format here; the RTDetrImageProcessor converts them to normalized center boxes at training time. Frames without tanks are kept as background examples.

In [ ]:
def tank_category_ids(coco):    ids = {category['id'] for category in coco.get('categories', []) if str(category.get('name', '')).lower() == 'tank'}    return ids or {1}def read_frames(json_path):    coco = json.loads(json_path.read_text())    category_ids = tank_category_ids(coco)    annotations = {}    for annotation in coco.get('annotations', []):        if annotation.get('category_id') in category_ids:            annotations.setdefault(annotation['image_id'], []).append(annotation)    frames = []    for image in coco.get('images', []):        image_path = json_path.parent / image['file_name']        if not image_path.is_file():            fallback = json_path.with_suffix('.png')            if fallback.is_file():                image_path = fallback        if not image_path.is_file():            print('missing image for', json_path.name)            continue        boxes = []        for annotation in annotations.get(image['id'], []):            x, y, w, h = annotation['bbox']            boxes.append([float(x), float(y), float(w), float(h)])        frames.append({            'image_path': image_path,            'width': float(image['width']),            'height': float(image['height']),            'boxes': boxes,        })    return framesframes = []for source in sources:    frames.extend(read_frames(source))if not frames:    raise SystemExit('Captures did not contain any usable frames.')random.Random(SEED).shuffle(frames)if len(frames) >= 2:    val_count = min(len(frames) - 1, max(1, round(len(frames) * VAL_FRACTION)))else:    val_count = 0splits = {'val': frames[:val_count], 'train': frames[val_count:]}if not splits['val']:    splits['val'] = splits['train']    print('Fewer than two frames — validation reuses the training frame.')for split, items in splits.items():    print(f'{split:5s}: {len(items):3d} frame(s), {sum(len(frame["boxes"]) for frame in items):4d} box(es)')

## 4. Preview the labels

In [ ]:
import numpy as npfrom PIL import Image, ImageDrawimport matplotlib.pyplot as pltdef annotated_preview(frame):    image = Image.open(frame['image_path']).convert('RGB')    draw = ImageDraw.Draw(image)    for x, y, w, h in frame['boxes']:        draw.rectangle([x, y, x + w, y + h], outline=(255, 90, 60), width=3)    return imagepreview_frames = (splits['train'] or splits['val'])[:4]figure, axes = plt.subplots(1, len(preview_frames), figsize=(5 * max(1, len(preview_frames)), 4))for axis, frame in zip(np.atleast_1d(axes), preview_frames):    axis.imshow(annotated_preview(frame))    axis.set_title(frame['image_path'].stem[-24:], fontsize=8)    axis.axis('off')plt.tight_layout()plt.show()

## 5. Load the processor and modelReplacing the 80 COCO labels with the single tank label reinitializes the classification head while keeping the pretrained backbone and decoder.

In [ ]:
from transformers import RTDetrForObjectDetection, RTDetrImageProcessorprocessor = RTDetrImageProcessor.from_pretrained(MODEL_ID)model = RTDetrForObjectDetection.from_pretrained(    MODEL_ID,    id2label=ID2LABEL,    label2id=LABEL2ID,    ignore_mismatched_sizes=True,)print(model.config.id2label)

## 6. Dataset and collateThe processor handles resizing to 640x640, normalization, and converting COCO boxes into the target format the model expects.

In [ ]:
class TankDataset(torch.utils.data.Dataset):    def __init__(self, frames, processor):        self.frames = frames        self.processor = processor    def __len__(self):        return len(self.frames)    def __getitem__(self, index):        frame = self.frames[index]        image = Image.open(frame['image_path']).convert('RGB')        annotations = [            {                'image_id': index,                'category_id': LABEL2ID['tank'],                'bbox': [x, y, w, h],                'area': w * h,                'iscrowd': 0,            }            for x, y, w, h in frame['boxes']        ]        encoding = self.processor(images=image, annotations=annotations, return_tensors='pt')        return {            'pixel_values': encoding['pixel_values'][0],            'labels': encoding['labels'][0],        }def collate_fn(batch):    return {        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),        'labels': [item['labels'] for item in batch],    }

## 7. Fine-tune with Trainer

In [ ]:
from transformers import Trainer, TrainingArgumentstrain_dataset = TankDataset(splits['train'], processor)val_dataset = TankDataset(splits['val'], processor)training_args = TrainingArguments(    output_dir=str(OUTPUT_DIR),    num_train_epochs=EPOCHS,    per_device_train_batch_size=BATCH,    per_device_eval_batch_size=BATCH,    learning_rate=LEARNING_RATE,    weight_decay=1e-4,    warmup_ratio=0.05,    lr_scheduler_type='cosine',    logging_steps=10,    save_strategy='epoch',    eval_strategy='no',    remove_unused_columns=False,    dataloader_num_workers=2,    fp16=torch.cuda.is_available(),    report_to='none',)trainer = Trainer(    model=model,    args=training_args,    train_dataset=train_dataset,    data_collator=collate_fn,)trainer.train()trainer.save_model(str(OUTPUT_DIR))processor.save_pretrained(str(OUTPUT_DIR))

## 8. EvaluateRT-DETR is NMS-free; torchmetrics computes mAP directly from the 300 query predictions and the exact game labels.

In [ ]:
from torchmetrics.detection import MeanAveragePrecisiondevice = next(model.parameters()).devicemodel.eval()metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')for frame in splits['val']:    image = Image.open(frame['image_path']).convert('RGB')    inputs = processor(images=image, return_tensors='pt')    with torch.no_grad():        outputs = model(pixel_values=inputs['pixel_values'].to(device))    results = processor.post_process_object_detection(        outputs,        target_sizes=torch.tensor([(image.height, image.width)], device=device),        threshold=0.0,    )[0]    ground_truth = torch.tensor(        [[x, y, x + w, y + h] for x, y, w, h in frame['boxes']],        dtype=torch.float32,    ).reshape(-1, 4)    metric.update(        [{            'boxes': results['boxes'].detach().cpu(),            'scores': results['scores'].detach().cpu(),            'labels': results['labels'].detach().cpu(),        }],        [{            'boxes': ground_truth,            'labels': torch.zeros(len(ground_truth), dtype=torch.long),        }],    )metrics = metric.compute()print(f"mAP50    : {float(metrics['map_50']):.4f}")print(f"mAP50-95 : {float(metrics['map']):.4f}")

## 9. Export ONNX for the browserOptimum does not yet cover RT-DETR export, so the notebook wraps the model and exports it with torch.onnx. The wrapper folds the decoder output into one fixed tensor:- Input: pixel_values float32 [1, 3, 640, 640], RGB resized to 640x640, scaled by 1/255, normalized with ImageNet mean/std.- Output: detections float32 [1, 300, 6] — x1, y1, x2, y2 in normalized 0..1 coordinates, sigmoid confidence, class id. No NMS.The game multiplies the corner coordinates by the current frame size before drawing.

In [ ]:
import torch.nn as nnclass BrowserDetector(nn.Module):    def __init__(self, detector):        super().__init__()        self.detector = detector    def forward(self, pixel_values):        outputs = self.detector(pixel_values=pixel_values)        probabilities = torch.sigmoid(outputs.logits)        scores, labels = probabilities.max(dim=-1)        cx, cy, width, height = outputs.pred_boxes.unbind(-1)        corners = torch.stack([            cx - width / 2,            cy - height / 2,            cx + width / 2,            cy + height / 2,        ], dim=-1)        return torch.cat([corners, scores.unsqueeze(-1), labels.to(torch.float32).unsqueeze(-1)], dim=-1)detector = BrowserDetector(model).to(device).eval()dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device)onnx_path = WORK / 'tank-detector.onnx'torch.onnx.export(    detector,    (dummy,),    str(onnx_path),    input_names=['pixel_values'],    output_names=['detections'],    opset_version=17,    do_constant_folding=True,)labels_path = WORK / 'tank-detector.labels.txt'labels_path.write_text('\n'.join(CLASSES) + '\n')print('model :', onnx_path)print('labels:', labels_path)

## 10. Verify the exported graph

In [ ]:
import onnxruntime as ortsession = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])input_meta = session.get_inputs()[0]output_meta = session.get_outputs()[0]print('input :', input_meta.name, input_meta.shape)print('output:', output_meta.name, output_meta.shape)dummy_input = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)detections = np.asarray(session.run([output_meta.name], {input_meta.name: dummy_input})[0])assert detections.shape == (1, 300, 6), detections.shapeprint('output tensor:', detections.shape)

## 11. Run the ONNX graph on a real capture

In [ ]:
def detect_onnx(image, confidence=0.25):    inputs = processor(images=image, return_tensors='pt')    raw = np.asarray(session.run([output_meta.name], {input_meta.name: inputs['pixel_values'].numpy()})[0])[0]    width, height = image.size    results = []    for x1, y1, x2, y2, score, class_id in raw:        if score < confidence:            continue        results.append((x1 * width, y1 * height, x2 * width, y2 * height, float(score), int(class_id)))    return resultscheck_frame = preview_frames[0]check_image = Image.open(check_frame['image_path']).convert('RGB')detections = detect_onnx(check_image)overlay = check_image.copy()draw = ImageDraw.Draw(overlay)for x1, y1, x2, y2, score, class_id in detections:    draw.rectangle([x1, y1, x2, y2], outline=(80, 255, 160), width=3)    draw.text((x1 + 4, max(0, y1 - 12)), f'{CLASSES[class_id]} {score:.2f}', fill=(80, 255, 160))print(f'{len(detections)} detection(s) at confidence >= 0.25')plt.figure(figsize=(10, 6))plt.imshow(overlay)plt.axis('off')plt.show()

## 12. Ship it to the game1. Download tank-detector.onnx and tank-detector.labels.txt from the notebook output.2. Put both in packages/game/public/models/ (the models folder is git-ignored).3. Keep CLASSES and tank-detector.labels.txt in sync — line order is the class-id mapping.The game draws software boxes today because it already knows every tank transform. This model is the pixel-only alternative used when detection is switched to the AI source. Section 8 reports validation mAP, and section 11 checks the exported graph on a recorded frame.The browser runtime must mirror the processor: resize the frame to 640x640, RGB, scale by 1/255, normalize with mean [0.485, 0.456, 0.406] and std [0.229, 0.224, 0.225], then read normalized boxes from the single detections output.